# 03 - Cleaning & Normalization Pipeline (Task 4)
**Owner:** Rabi Sandhu

All code is in `src/clean.py` -> one function `clean_text()`.

In [10]:
import sys, re
from pathlib import Path
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
df = pd.read_csv(ROOT / "data/processed/sample_raw.csv")
print(df.shape)

(40000, 6)


In [11]:
import clean
from clean import clean_text, normalize_urdu, remove_noise, light_stem, URDU_STOPWORDS, NEGATIONS

## Step-by-step on one raw-style example

In [12]:
raw = ("اسلام آباد: عالمی بینک نے ۲۰ لاکھ ڈالر فراہم کرنے کا اعلان کیا، "
       "جس سے متاثرہ خاندانوں کی مدد ہوگی۔ برآمدات میں اضافہ نہیں ہوا۔ <p>Dawn</p> https://dawn.com كتاب")
s1 = normalize_urdu(raw);          print("1-2 NFC + Urdu norm :", s1)
s2 = remove_noise(s1);             print("3   noise removed   :", s2)
s3 = clean.NUMBER.sub(" NUM ", s2.lower()); print("4-5 case + numbers  :", s3)
t = s3.split();                    print("6   tokens          :", t)
t = [w for w in t if w not in URDU_STOPWORDS]; print("7   no stop words   :", t)
t = [light_stem(w) for w in t];    print("8   stemmed         :", t)
print("\nclean_text() gives  :", clean_text(raw))

1-2 NFC + Urdu norm : اسلام آباد: عالمی بینک نے 20 لاکھ ڈالر فراہم کرنے کا اعلان کیا، جس سے متاثرہ خاندانوں کی مدد ہوگی۔ برآمدات میں اضافہ نہیں ہوا۔ <p>Dawn</p> https://dawn.com کتاب
3   noise removed   : اسلام آباد عالمی بینک نے 20 لاکھ ڈالر فراہم کرنے کا اعلان کیا جس سے متاثرہ خاندانوں کی مدد ہوگی برآمدات میں اضافہ نہیں ہوا کتاب
4-5 case + numbers  : اسلام آباد عالمی بینک نے  NUM  لاکھ ڈالر فراہم کرنے کا اعلان کیا جس سے متاثرہ خاندانوں کی مدد ہوگی برآمدات میں اضافہ نہیں ہوا کتاب
6   tokens          : ['اسلام', 'آباد', 'عالمی', 'بینک', 'نے', 'NUM', 'لاکھ', 'ڈالر', 'فراہم', 'کرنے', 'کا', 'اعلان', 'کیا', 'جس', 'سے', 'متاثرہ', 'خاندانوں', 'کی', 'مدد', 'ہوگی', 'برآمدات', 'میں', 'اضافہ', 'نہیں', 'ہوا', 'کتاب']
7   no stop words   : ['اسلام', 'آباد', 'عالمی', 'بینک', 'NUM', 'لاکھ', 'ڈالر', 'فراہم', 'اعلان', 'متاثرہ', 'خاندانوں', 'مدد', 'برآمدات', 'اضافہ', 'نہیں', 'کتاب']
8   stemmed         : ['اسلام', 'آباد', 'عالمی', 'بینک', 'NUM', 'لاکھ', 'ڈالر', 'فراہم', 'اعلان', 'متاثرہ', 'خاندان', 'مد

## Urdu normalization examples

In [13]:
for a in ["كتاب", "يہ", "۲۰۲۴", "رکھتی ہےفنانس", "متاثر ہیںیہ بھی"]:
    print(repr(a), "->", repr(normalize_urdu(a)))

'كتاب' -> 'کتاب'
'يہ' -> 'یہ'
'۲۰۲۴' -> '2024'
'رکھتی ہےفنانس' -> 'رکھتی ہے فنانس'
'متاثر ہیںیہ بھی' -> 'متاثر ہیں یہ بھی'


**Glued words fix:** in Urdu, `ے` and `ں` only come at the END of a word. If a letter comes right after them, two words were glued
(because the dataset authors removed the full stops). So we add a space there: `ہےفنانس -> ہے فنانس`. This fixes most glued sentences.

In [14]:
import re
docs_raw = df["News Text"].tolist()
before = sum(len(clean.GLUED.findall(d)) for d in docs_raw)
print("Glued words found and split in the sample:", before)

Glued words found and split in the sample: 67333


## Stop words - negations are kept

In [15]:
print("Stop words:", len(URDU_STOPWORDS))
print("Negations kept:", NEGATIONS, "| any in stop list?", bool(NEGATIONS & URDU_STOPWORDS))
print(clean_text("قیمتوں میں اضافہ ہوا"), "  vs  ", clean_text("قیمتوں میں اضافہ نہیں ہوا"))

Stop words: 115
Negations kept: {'نہیں', 'مت', 'بغیر', 'نہ'} | any in stop list? False
قیمت اضافہ   vs   قیمت اضافہ نہیں


## Stemming examples (light suffix stemmer)

In [16]:
words = ["خاندانوں", "شہریوں", "کمپنیاں", "کتابیں", "ملازمین", "منصوبوں", "معاملات", "خواتین", "باتیں"]
pd.DataFrame({"word": words, "stem": [light_stem(w) for w in words]})

,word,stem
0,خاندانوں,خاندان
1,شہریوں,شہری
2,کمپنیاں,کمپنی
3,کتابیں,کتاب
4,ملازمین,ملازمین
5,منصوبوں,منصوب
6,معاملات,معامل
7,خواتین,خواتین
8,باتیں,بات


**Why stemming and not lemmatization:** NLTK and spaCy have no Urdu lemmatizer (it needs a dictionary + POS tags).
A light stemmer merges plural forms (`خاندانوں -> خاندان`, `کمپنیاں -> کمپنی`) and is enough for topic classification.
Downside: some words are over-stemmed (`منصوبوں -> منصوب`, `معاملات -> معامل`) and Arabic plurals (`خواتین`, `ملازمین`) are not handled.

## Vocabulary before vs after (whole sample)

In [17]:
docs = (df["Headline"].fillna("") + " " + df["News Text"].fillna("")).tolist()
cleaned = [clean_text(d) for d in docs]
vb = Counter(w for d in docs for w in d.split()); va = Counter(w for d in cleaned for w in d.split())
pd.DataFrame({"tokens": [sum(vb.values()), sum(va.values())], "types (vocab)": [len(vb), len(va)]},
             index=["before", "after"])

,tokens,types (vocab)
before,9017589,204048
after,5647221,135935


In [18]:
for d, c in list(zip(docs, cleaned))[:3]:
    print("BEFORE:", d[:150]); print("AFTER :", c[:150]); print()

BEFORE: سری لنکا کے خلاف سیریز کے لئے قومی خواتین ٹیم کا اعلان لاہور ویب ڈیسک پاکستان کرکٹ بورڈ نے جنوری سے متحدہ امرات میں سری لنکا کے خلاف ہونے والی سیریز ک
AFTER : سری لنکا خلاف سیریز قومی خواتین ٹیم اعلان لاہور ویب ڈیسک پاکستان کرکٹ بورڈ جنوری متحدہ امر سری لنکا خلاف سیریز قومی خواتین کرکٹ ٹیم اعلان کردیا تفصیل 

BEFORE: محمد عامر نے انٹرنیشنل کرکٹ میں واپسی کیلئے ٹی ٹوئنٹی ورلڈ کپ کو ہدف بنا لیا فیصل باداردو پوائنٹ تازہ ترین اخبار 12 مئی 2015 فاسٹ بالر محمد عامر نے ان
AFTER : محمد عامر انٹرنیشنل کرکٹ واپسی ٹی ٹوئنٹی ورلڈ کپ ہدف بنا لیا فیصل باداردو پوائنٹ تازہ ترین اخبار NUM مئی NUM فاسٹ بالر محمد عامر انٹرنیشنل کرکٹ واپسی 

BEFORE: جاپان میں 32 ویں سالانہ ٹوکیو انٹرنیشنل فلم فیسٹیول شروع ٹوکیو اردو پوائنٹ اخبارتازہ ترین اے پی پی 29 اکتوبر2019ء جاپان کے درالحکومت ٹوکیو میں ایشیا ک
AFTER : جاپان NUM ویں سالانہ ٹوکیو انٹرنیشنل فلم فیسٹیول شروع ٹوکیو اردو پوائنٹ اخبارتازہ ترین اے پی پی NUM اکتوبر NUM جاپان درالحکومت ٹوکیو ایشیا سب بڑے فلمی

